# Single Runtime Runbook - Recurrent Qwen

Keep one Colab runtime attached and move through the maintained target queue by changing `TARGET`. This notebook deliberately does not route to separate notebooks. It fetches the current bootstrap from GitHub by resolved commit SHA and executes one explicit `STAGE5_CURRENT_A100_TARGET`.

Current queue: `reentry_repair_smoke` -> review -> `reentry_recovery_training` -> `debiased_benchmark_suite` -> `dense_mcq_trace_sft_control`.

Use `master_sequence_status` first if a runtime was restarted and you only want a cheap status readout.


In [ ]:
import base64, json, os, time, urllib.request
from google.colab import userdata

REPO = "mshapiro123/recurrent-qwen-svgd"
TARGET = "reentry_repair_smoke"
# Other maintained targets:
# TARGET = "master_sequence_status"
# TARGET = "reentry_recovery_training"
# TARGET = "debiased_benchmark_suite"
# TARGET = "dense_mcq_trace_sft_control"

gh = userdata.get("GH_TOKEN") or userdata.get("GITHUB_TOKEN")
assert gh, "Missing GH_TOKEN or GITHUB_TOKEN in Colab secrets."

hf = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_HUB_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf

os.environ["STAGE5_CURRENT_A100_TARGET"] = TARGET

headers = {
    "Authorization": f"Bearer {gh}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "Cache-Control": "no-cache",
}

def gh_json(url):
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.load(response)

resolved_ref = gh_json(
    f"https://api.github.com/repos/{REPO}/git/refs/heads/main?cache_bust={time.time_ns()}"
)["object"]["sha"]

payload = gh_json(
    f"https://api.github.com/repos/{REPO}/contents/colab/CURRENT_A100_BOOTSTRAP_CELL.py"
    f"?ref={resolved_ref}&cache_bust={time.time_ns()}"
)

code = base64.b64decode(payload["content"]).decode("utf-8")
required = [
    "sha_resolved_nested_fetch_v3",
    TARGET,
    "STAGE5_CURRENT_A100_TARGET",
]
missing = [marker for marker in required if marker not in code]
assert not missing, f"Fetched stale or incomplete bootstrap: {missing}"
print("Fetched bootstrap sha:", payload.get("sha"), "commit:", resolved_ref[:12], "target:", TARGET)
exec(compile(code, "colab/CURRENT_A100_BOOTSTRAP_CELL.py", "exec"))


After a target publishes, do not guess the next run. Either run `TARGET = "master_sequence_status"` or inspect `colab/review_stage5_reentry.py --no_write` / `colab/print_current_stage5_action.py` from the cloned repo. The current phase order is documented in `docs/PROGRAM_TRACK_MASTER_SEQUENCE.md` and `colab/NEXT_COLAB_SEQUENCE.md`.
